In [6]:
%pip install python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import json
import os
from google import genai
from google.genai import types
from dotenv import load_dotenv  # 1. Import dotenv

# 2. Load the variables from your .env file into os.environ
load_dotenv() 

# 3. Now the client will successfully find the key
client = genai.Client()
MODEL_ID = "gemini-2.5-flash"

# =====================================================================
# 1. Define the Python Functions (Tools)
# =====================================================================

def lookup_order(order_id: str) -> str:
    """Looks up order details (item, price, purchase date, warranty_months) from orders.json by ID."""
    print(f"⚙️ [Tool Execution] Running lookup_order for: {order_id}")
    try:
        with open("orders.json", "r") as f:
            orders = json.load(f)
        
        if order_id in orders:
            return json.dumps(orders[order_id])
        return json.dumps({"error": f"Order {order_id} not found."})
    except Exception as e:
        return json.dumps({"error": f"Database error: {str(e)}"})

def calculate(expression: str) -> str:
    """Evaluates a basic mathematical or arithmetic expression string safely."""
    print(f"⚙️ [Tool Execution] Running calculate for: {expression}")
    try:
        # Safe evaluation restricted to basic math symbols
        allowed_chars = "0123456789+-*/(). "
        if not all(char in allowed_chars for char in expression):
            return json.dumps({"error": "Invalid characters in expression."})
        
        result = eval(expression)
        return json.dumps({"result": float(result)})
    except Exception as e:
        return json.dumps({"error": f"Calculation error: {str(e)}"})

# Map string names to actual executable functions for our execution loop
available_tools = {
    "lookup_order": lookup_order,
    "calculate": calculate
}

# =====================================================================
# 2. Define the Orchestration Loop
# =====================================================================

def run_agent_conversation(prompt: str):
    print(f"\n🚀 User Prompt: '{prompt}'")
    print("-" * 60)

    # Initialize a fresh chat session for each prompt containing our tools config
    chat = client.chats.create(
        model=MODEL_ID,
        config=types.GenerateContentConfig(
            tools=[lookup_order, calculate], # Python functions passed directly as tools
            temperature=0.0                  # Lower temp for deterministic tool routing
        )
    )

    # Send the initial user message
    response = chat.send_message(prompt)

    # Multi-turn execution loop: Continue as long as the model requests functions
    while response.function_calls:
        tool_responses = []

        for call in response.function_calls:
            print(f"🤖 [Model Request] Tool Call: {call.name} | Args: {call.args}")
            
            # Extract names and arguments
            tool_name = call.name
            tool_args = dict(call.args)

            if tool_name in available_tools:
                # Execute the native python function
                execution_result = available_tools[tool_name](**tool_args)
                
                # Format the response back to the model's schema
                tool_responses.append(
                    types.Part.from_function_response(
                        name=tool_name,
                        response={"result": execution_result}
                    )
                )
            else:
                print(f"❌ Error: Model requested unknown tool '{tool_name}'")
                return

        # Send the tool execution output back to the model
        response = chat.send_message(tool_responses)

    # Loop ends when response.function_calls is empty; print final textual answer
    print(f"🏁 Final Answer:\n{response.text}\n")


# =====================================================================
# 3. Running the Test Cases
# =====================================================================
if __name__ == "__main__":
    if not os.environ.get("GEMINI_API_KEY"):
        print("Please export your GEMINI_API_KEY environment variable first!")
        exit(1)

    # Scenario 1: No tool required
    run_agent_conversation("What can you help me with?")

    # Scenario 2: Two tools sequentially chained
    run_agent_conversation("For order A1001, what would the total be if I bought three of them?")

    # Scenario 3: Stretch goal - graceful handling of a bad lookup
    run_agent_conversation("What is the status of order A9999?")


🚀 User Prompt: 'What can you help me with?'
------------------------------------------------------------
🏁 Final Answer:
I can help you with the following:

* **Look up order details:** Provide me with an order ID, and I can retrieve information like the item, price, purchase date, and warranty months.
* **Calculate mathematical expressions:** I can evaluate basic mathematical or arithmetic expressions.


🚀 User Prompt: 'For order A1001, what would the total be if I bought three of them?'
------------------------------------------------------------
⚙️ [Tool Execution] Running lookup_order for: A1001
⚙️ [Tool Execution] Running calculate for: 1200 * 3
🏁 Final Answer:
The total for three of order A1001 would be 3600.


🚀 User Prompt: 'What is the status of order A9999?'
------------------------------------------------------------
⚙️ [Tool Execution] Running lookup_order for: A9999
🏁 Final Answer:
I'm sorry, but I cannot find the order A9999.

